# RQ4 on Google Colab — remote Linux/CUDA execution (Stage 1 pilot → reviewed Stage 2)

**Read this first.**
* This notebook runs the *frozen* RQ4 protocol (`FROZEN_PROTOCOL.md`, unchanged) on a **hosted Colab GPU runtime**. It must **not** be run on a local runtime or on a Mac; cell 1 refuses to continue there, and the runner itself refuses inference on macOS.
* **No results exist yet.** Until you run it, there are zero measured rows. A Stage 1 pilot is `PILOT_ONLY` and must never be reported as the ten-item result.
* **Colab is not always-on.** All model work runs on the remote VM, so your Mac does not need to stay powered on. Free Colab can disconnect or recycle the runtime, caps session length, and does **not guarantee** a GPU (you may be told none is available — then try later). A browser disconnect can interrupt a run; do not rely on background execution. Reconnect and resume from Drive checkpoints.
* **Resuming:** every evaluation is checkpointed atomically to your Google Drive. After any disconnect, reconnect and re-run cells 1–7 (setup; safe to re-run), then re-run the Stage 1 or Stage 2 cell: finished evaluations are skipped, and a checkpoint written under a different code commit/config is rejected rather than mixed.
* **Model weights are not stored on Drive.** They are downloaded to the remote VM's disk each session (public files, no login or token needed). Only results/checkpoints go to Drive. Do not paste any secrets into this notebook.
* **Drive authorization:** cell 5 asks you to authorize access to your own Google Drive. Do this yourself in the pop-up. Nothing runs in the background after a Colab session ends.

## 1. Verify a hosted Colab runtime with a CUDA GPU

**Free compute only.** Select a GPU only if one is available under your existing free / no-new-payment Colab allocation (e.g. a free T4). If none is offered, or Colab prompts you to buy compute units, upgrade, or pick a premium GPU (A100, L4, H100, TPU, Pro/Pro+), **stop** and retry later. Never pay or switch plans for this notebook, and do not select a premium GPU.


In [ ]:
import os, sys, subprocess, shutil

assert sys.platform == "linux", f"Not a Linux VM ({sys.platform}); this notebook is remote-only."
HOSTED = "COLAB_RELEASE_TAG" in os.environ and os.path.isdir("/content")
if not HOSTED:
    raise SystemExit("This does not look like a HOSTED Colab runtime (possibly a local runtime). Refusing. "
                     "Use Runtime > Change runtime type > a hosted GPU runtime ONLY if a free (no-new-payment) GPU such as a T4 is offered; never buy compute or choose a premium GPU.")
if shutil.which("nvidia-smi") is None:
    raise SystemExit("No GPU visible (nvidia-smi missing). Runtime > Change runtime type > a FREE GPU (T4) only if offered, then reconnect and re-run. "
                     "GPUs are not guaranteed on free Colab: if none is available, STOP and try again later. Do NOT buy compute or pick a premium GPU. Do NOT run on CPU.")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise SystemExit("nvidia-smi found no usable GPU. Do NOT run on CPU. " + gpu.stderr)
print("GPU:", gpu.stdout.strip())
print(subprocess.run(["free", "-h"], capture_output=True, text=True).stdout)
print("Python", sys.version.split()[0])
if sys.version_info[:2] != (3, 12):
    raise SystemExit("The pinned stack was tested on Python 3.12; this runtime has a different minor version. Stop and ask the auditor.")

## 2. Pin the exact code commit
`CODE_COMMIT` is filled in by the auditor after the code commit. The runner refuses to run if it is still the placeholder, is not a full 40-hex SHA, or does not equal the checkout's `HEAD`; the commit is also part of the resume fingerprint.

In [ ]:
import re
CODE_COMMIT = "__RQ4_CODE_COMMIT__"
if not re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT):
    raise SystemExit("CODE_COMMIT is still the placeholder (or not a full 40-hex SHA). Wait for the auditor to pin it; do not guess.")
os.environ["RQ4_CODE_COMMIT"] = CODE_COMMIT
print("code commit:", CODE_COMMIT)

## 3. Clone the repository and check out that exact commit (idempotent)

In [ ]:
REPO_URL = "https://github.com/MelodyResearch/j-lens-bias.git"
REPO_DIR = "/content/j-lens-bias"
def sh(*a, check=True, cwd=None):
    r = subprocess.run(list(a), capture_output=True, text=True, cwd=cwd)
    if check and r.returncode != 0:
        raise SystemExit(f"command failed: {' '.join(a)}\n{r.stdout}\n{r.stderr}")
    return r.stdout.strip()

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh("git", "clone", "--no-checkout", REPO_URL, REPO_DIR)
sh("git", "fetch", "--all", "--tags", cwd=REPO_DIR)
if subprocess.run(["git", "cat-file", "-e", CODE_COMMIT + "^{commit}"], cwd=REPO_DIR).returncode != 0:
    sh("git", "fetch", "origin", CODE_COMMIT, cwd=REPO_DIR)
sh("git", "checkout", "--detach", CODE_COMMIT, cwd=REPO_DIR)
head = sh("git", "rev-parse", "HEAD", cwd=REPO_DIR)
assert head == CODE_COMMIT, f"HEAD {head} != {CODE_COMMIT}"
RQ4_DIR = os.path.join(REPO_DIR, "Week 4", "submissions", "MelodyResearch_rq4")
assert os.path.isfile(os.path.join(RQ4_DIR, "run_rq4.py"))
print("checked out", head)

## 4. Install the pinned dependencies in the remote runtime
This installs only free public packages; it does not need or authorize any paid compute. If Colab offers to buy compute units or upgrade, decline.


In [ ]:
# Free public packages only; no paid compute or upgrade is needed or authorized here.
pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(RQ4_DIR, "requirements-colab.txt")],
                     capture_output=True, text=True)
print(pip.stdout[-1500:], pip.stderr[-1500:])
assert pip.returncode == 0, "pip install failed (see output above)"
# Verify in a fresh subprocess (this kernel never imports torch/transformers/jlens).
chk = subprocess.run([sys.executable, "-c",
    "import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"],
    capture_output=True, text=True)
print(chk.stdout, chk.stderr[-500:])
tv, trv, cuda_ok, *_ = chk.stdout.split()
assert (tv.split("+")[0], trv) == ("2.14.1", "5.20.2"), "pinned versions not active - Runtime > Restart session, then re-run from cell 1"
assert cuda_ok == "True", "torch cannot see the GPU"

## 5. Mount Google Drive for persistent checkpoints (requires **your** authorization)
Results and checkpoints go to `MyDrive/j-lens-bias-rq4/results_<commit>` so they survive a Colab timeout. A separate folder per code commit means a different commit can never be mixed with these checkpoints.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
mounted = any(l.split()[1] == "/content/drive" and l.split()[2].startswith("fuse") for l in open("/proc/mounts"))
assert mounted, "Drive is not mounted (authorization not completed?). Re-run this cell and authorize."
RESULTS_DIR = f"/content/drive/MyDrive/j-lens-bias-rq4/results_{CODE_COMMIT[:12]}"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ["RQ4_RESULTS_DIR"] = RESULTS_DIR
print("persistent results dir (also visible in the Drive web UI):", RESULTS_DIR)

## 6. Hugging Face cache on the remote VM disk (never Drive) and public model/lens download

In [ ]:
import json
HF_HOME = "/content/hf_home"          # ephemeral VM disk; re-downloaded every session on purpose
assert not HF_HOME.startswith("/content/drive")
os.environ["HF_HOME"] = HF_HOME
os.environ["PYTHONUNBUFFERED"] = "1"
EXPECTED = {   # recorded in EXECUTION_STATUS.md (Stage 0)
    "model_commit": "2fc06364715b967f1860aea9cf38778875588b17",   # exact repo commits, also pinned in rq4/config.py
    "lens_commit": "16a01f309fcec900fdcec3f4cd5b64f3d00e4d5a",
    "model_sha256": "04b1c301231dd422b8860db31311ab2721511346a32cb1e079c4c4e5f1fe4696",
    "lens_sha256": "aa26b68ed73cf903280dbd8d1806f4ed8580aad205f396a5c997ee19259c9b48",
}
dl = r"""
import hashlib, json, sys
sys.path.insert(0, sys.argv[1])
from pathlib import Path
from huggingface_hub import snapshot_download, hf_hub_download
from rq4 import config as C
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 24), b""):
            h.update(b)
    return h.hexdigest()
assert C.MODEL_REVISION == sys.argv[2] and C.LENS_REVISION == sys.argv[3], "config revisions differ from the notebook pins"
m = Path(snapshot_download(C.MODEL_NAME, revision=C.MODEL_REVISION))
l = Path(hf_hub_download(C.LENS_REPO, C.LENS_FILE, revision=C.LENS_REVISION))
shards = sorted(m.glob("*.safetensors"))
out = {"model_commit": m.name, "lens_commit": l.parts[l.parts.index("snapshots") + 1],
       "model_sha256": sha(shards[0]) if len(shards) == 1 else None, "lens_sha256": sha(l), "n_model_shards": len(shards)}
print(json.dumps(out))
"""
r = subprocess.run([sys.executable, "-c", dl, RQ4_DIR, EXPECTED["model_commit"], EXPECTED["lens_commit"]], capture_output=True, text=True)
print(r.stderr[-1500:])
got = json.loads(r.stdout.strip().splitlines()[-1]) if r.returncode == 0 else None
assert got, "download failed"
print(got)
assert got["n_model_shards"] == 1, f"expected exactly one model safetensors shard, got {got['n_model_shards']}"
for k, v in EXPECTED.items():
    assert got.get(k) == v, f"{k} mismatch: {got.get(k)} != {v}"

## 7. Prepare-only check and synthetic tests (no model; safe to re-run)

In [ ]:
def run(args, cwd=RQ4_DIR, env=None):
    p = subprocess.Popen([sys.executable, *args], cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="")
    return p.wait()

assert run(["run_rq4.py"]) == 0                       # default = prepare-only
assert run(["-m", "pytest", "tests", "-q"], env={"RQ4_RESULTS_DIR": "", "RQ4_CODE_COMMIT": ""}) == 0   # synthetic tests only; results env cleared for them

## 8. Stage 1 pilot — ONE base item × 4 conditions × 2 formats (8 prompts, 128 evaluations) — `PILOT_ONLY`
This loads the model **on the remote GPU** after three memory/VRAM readings 25 s apart (≥ 60 % available RAM, enough free VRAM, low pressure). If a reading fails, the run stops *before* the model is loaded; just wait and re-run. Dtype (bf16 on native-bf16 GPUs, otherwise fp16 on e.g. T4) and library versions are recorded and pinned for later sessions.

In [ ]:
rc = run(["run_rq4.py", "--run-inference", "--i-confirm-no-heavy-experiment", "--stage", "1"])
print("runner exit code:", rc, "(non-zero = stopped safely; completed checkpoints are kept on Drive; re-run this cell to resume)")

### 8b. Objective, outcome-blind pilot validation and the raw generations

In [ ]:
valid_rc = run(["run_rq4.py", "--validate-stage1"])
print("\nvalidator exit code:", valid_rc, "(0 = PASS: eligible for review; it is NOT approval)")
print("\n===== RAW GENERATIONS (read all of them) =====")
run(["run_rq4.py", "--show-stage1-generations"])

## 9. Review acknowledgement (human / Codex) — required before Stage 2
A validator PASS only means the structure and safety checks held. Before Stage 2 someone must **read the raw generations above** (and the memory log / preflight JSON in the results folder) and confirm they are sane. Fill in the fields yourself; leave `ACK = False` until you have actually done so. The CLI re-checks that the validator passes before recording the approval.

In [ ]:
REVIEWER = ""                 # your name
REVIEW_NOTE = ""              # what you read, e.g. "read all 128 raw generations, memory log, runtime_preflight.json; <findings>"
ACK = False                   # set True only after the review
if not ACK:
    print("Stage 2 stays LOCKED (ACK is False).")
else:
    assert REVIEWER.strip() and len(REVIEW_NOTE.strip()) >= 20, "fill REVIEWER and REVIEW_NOTE"
    assert run(["run_rq4.py", "--approve-stage1", REVIEW_NOTE, "--reviewer", REVIEWER]) == 0
    print("Approval recorded in", RESULTS_DIR)

## 10. Stage 2 — the frozen ten-item set (1,280 evaluations), resumable across sessions
Allowed only after (a) an objective validator PASS, (b) the acknowledgement above (stored on Drive as `STAGE1_APPROVED`, so it survives sessions), and (c) unchanged pilot checkpoints under the same code commit — the runner re-checks all three. Stage 1 rows are reused by key. This can take several Colab sessions: if the runtime disconnects, reconnect, re-run cells 1–7, then re-run this cell. A safety-excluded cell is recorded faithfully and never replaced.

In [ ]:
approved = os.path.isfile(os.path.join(RESULTS_DIR, "STAGE1_APPROVED"))   # persisted on Drive by cell 9 (possibly in an earlier session)
if not approved:
    print("Stage 2 LOCKED: needs an objective validator PASS and the recorded review acknowledgement (cell 9). Nothing was run.")
else:
    rc = run(["run_rq4.py", "--run-inference", "--i-confirm-no-heavy-experiment", "--stage", "2"])
    print("runner exit code:", rc, "- re-run this cell (after cells 1-7 in a new session) until it reports nothing left to do.")

## 11. Analyze saved results and show figures (reads only saved results; no model)

In [ ]:
run(["run_rq4.py", "--combine"])
run(["run_rq4.py", "--analyze"])
print("\nResults folder (Drive):", RESULTS_DIR)
for sub in ("evals", "tables", "figures"):
    d = os.path.join(RESULTS_DIR, sub)
    print(f"  {sub}/:", len(os.listdir(d)) if os.path.isdir(d) else "(not created yet)")
print("  rq4_raw_results.jsonl, rq4_memory_log.jsonl, runtime_preflight.json, runtime_provenance.json, stage1_validation.json")
figdir = os.path.join(RESULTS_DIR, "figures")
if os.path.isdir(figdir):
    from IPython.display import Image, display
    for f in sorted(os.listdir(figdir)):
        if f.endswith(".png"):
            print(f); display(Image(os.path.join(figdir, f)))
else:
    print("No figures: there are no saved results yet (nothing is simulated).")

### Reading the outcome
* Only 8 prompts (one item) ⇒ `PILOT_ONLY`: do not interpret effects. Only the full ten-item run can answer RQ4, and even then with ten base items intervals are descriptive.
* The completion label (`COMPLETE_GATE_MET`, `PILOT_ONLY`, `INCONCLUSIVE`, `BLOCKED`) is printed by the analysis step above.
* Copy results out of Drive only after the run; the code commit and runtime provenance are in `runtime_provenance.json`.